In [1]:
import pandas as pd 
from sqlalchemy import create_engine

df=pd.read_csv("crime_data_clean.csv")


In [1]:
from sqlalchemy import create_engine
username= 'postgres'
password= 'your_password_here'
host= 'localhost'
port= '5432'
database= 'crime_analytics'

engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}')
with engine.connect() as conn:
    print("Successfull connection establsished")

Successfull connection establsished


In [3]:
df.to_sql('crime_data', engine, if_exists='replace', index=False)
print("data loaded")

data loaded


In [4]:
result = pd.read_sql('SELECT * FROM crime_data LIMIT 5', engine)
result

,city,title,text,crime_category,total_victims,murder_reason,murder_child_victims,murder_male_adult_victims,murder_female_adult_victims,kidnap_child_victims,kidnap_male_adult_victims,kidnap_female_adult_victims,crime_against_women_type,total_adult_victims,total_child_victims
0,Ghaziabad,"Minor molests mute girl, sent to observation home",GHAZIABAD: A 12-year-old boy was sent to an ob...,Crime Against Women,3,Not Applicable,0,0,0,0,0,0,"8,10",0,3
1,Ghaziabad,Family out to buy car robbed of Rs 1 lakh by b...,GHAZIABAD: A family that had visited an automo...,Unclassified,0,Not Applicable,0,0,0,0,0,0,Not Applicable,0,0
2,Ghaziabad,"This gang posed as cops to extort youths, couples",GHAZIABAD: Five persons were arrested on Monda...,Unclassified,0,Not Applicable,0,0,0,0,0,0,Not Applicable,0,0
3,Ghaziabad,"Residents object to loud music, thrashed in Gh...","Ghaziabad: Four tenants, including two soldier...",Unclassified,0,Not Applicable,0,0,0,0,0,0,Not Applicable,0,0
4,Ghaziabad,Man kills self after being refused money for d...,GHAZIABAD: A 25-year-old allegedly committed s...,Unclassified,0,Not Applicable,0,0,0,0,0,0,Not Applicable,0,0


In [5]:
count_result = pd.read_sql('SELECT COUNT(*) FROM crime_data', engine)
count_result

,count
0,466


In [6]:
query= """
SELECT city,crime_category, COUNT(*) as case_count
FROM crime_data
GROUP BY city,crime_category
ORDER BY case_count DESC;
"""
RESULT = pd.read_sql(query, engine)
RESULT


,city,crime_category,case_count
0,Lucknow,Crime Against Women,81
1,Ghaziabad,Crime Against Women,68
2,Lucknow,Murder,63
3,Lucknow,Unclassified,53
4,Ghaziabad,Murder,47
5,Kanpur,Crime Against Women,46
6,Ghaziabad,Unclassified,41
7,Kanpur,Murder,37
8,Kanpur,Unclassified,12
9,Ghaziabad,Kidnapping,11


In [7]:
query = """
SELECT city, murder_reason, COUNT(*) AS case_count
FROM crime_data
WHERE murder_reason != 'Not Applicable'
GROUP BY city, murder_reason
ORDER BY city, case_count DESC;
"""

result = pd.read_sql(query, engine)
result

,city,murder_reason,case_count
0,Ghaziabad,Unknown/Other,12
1,Ghaziabad,Petty Quarrels,11
2,Ghaziabad,Family Dispute,9
3,Ghaziabad,Love Affairs,6
4,Ghaziabad,Property Disputes,4
5,Ghaziabad,Money Disputes,3
6,Ghaziabad,Personal Vendetta,2
7,Kanpur,Unknown/Other,11
8,Kanpur,Love Affairs,8
9,Kanpur,Petty Quarrels,5


In [8]:
from sqlalchemy import text

fix_query = """
UPDATE crime_data
SET murder_reason = CASE
    WHEN murder_reason = 'Unknown/Other' THEN 'Unknown/other'
    WHEN murder_reason = 'Property Disputes' THEN 'Property/Land Disputes'
    ELSE murder_reason
END;
"""

with engine.connect() as conn:
    conn.execute(text(fix_query))
    conn.commit()

print("Fixed murder_reason labels in Postgres.")

Fixed murder_reason labels in Postgres.


In [9]:
query = """
SELECT DISTINCT murder_reason
FROM crime_data
ORDER BY murder_reason;
"""

result = pd.read_sql(query, engine)
result

,murder_reason
0,Casteism
1,Family Dispute
2,Love Affairs
3,Money Disputes
4,Not Applicable
5,Personal Vendetta
6,Petty Quarrels
7,Property/Land Disputes
8,Unknown/other


In [10]:
query = """
SELECT city,
       SUM(total_victims) AS total_victims,
       ROUND(AVG(total_victims), 2) AS avg_victims_per_case
FROM crime_data
GROUP BY city
ORDER BY total_victims DESC;
"""

result = pd.read_sql(query, engine)
result

,city,total_victims,avg_victims_per_case
0,Lucknow,171.0,0.85
1,Ghaziabad,144.0,0.86
2,Kanpur,102.0,1.05


In [2]:
sql_queries = """
-- Crime category breakdown by city
SELECT city, crime_category, COUNT(*) AS case_count
FROM crime_data
GROUP BY city, crime_category
ORDER BY city, case_count DESC;

-- Murder reason breakdown by city
SELECT city, murder_reason, COUNT(*) AS case_count
FROM crime_data
WHERE murder_reason != 'Not Applicable'
GROUP BY city, murder_reason
ORDER BY city, case_count DESC;

-- Total and average victims by city
SELECT city,
       SUM(total_victims) AS total_victims,
       ROUND(AVG(total_victims), 2) AS avg_victims_per_case
FROM crime_data
GROUP BY city
ORDER BY total_victims DESC;
"""

with open('crime_analysis_queries.sql', 'w') as f:
    f.write(sql_queries)

print("Saved: crime_analysis_queries.sql")

Saved: crime_analysis_queries.sql
